# Module 04.04 — Lens Visualizations (`lens`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.4 Lens Visualizations (`lens`)

Lens is the **modern, drag-and-drop** visualization builder introduced in Kibana 7.5.
It replaced the older aggregation-based flow for most chart types and is the recommended
path for all new visualizations. Unlike legacy visualizations, Lens always uses the `lens`
saved object type, so the two are easy to distinguish in the API.

The schema is considerably richer than the `visualization` type. Instead of an opaque
`visState` blob, Lens stores a structured `state` object with:
- `visualization` — the chart type and its per-layer rendering configuration
- `query` — a global filter applied on top of the layer queries
- `filters` — pinned filters from the Kibana filter bar
- `datasourceStates.formBased.layers` — each layer's field selections, aggregations,
  and column formulas

The `references` array links each layer to its Data View by ID.  Because the structure
is machine-readable, Kibana can migrate Lens objects between versions more reliably
than it can migrate `visState` blobs.

### Create in Kibana UI
1. Go to **Visualize Library → Create visualization → Lens**
2. Select the eCommerce data view
3. Drag `taxful_total_price` to the Y-axis (it auto-suggests Sum)
4. Drag `order_date` to the X-axis (it auto-suggests Date histogram)
5. Save as `Training — Lens Revenue Chart`

In [ ]:
heading('4.4 Lens Visualization — inspect sample data objects')

lens_objs = find_saved_objects('lens')
console.print(f'  Found {len(lens_objs)} lens object(s)')
if lens_objs:
    l = lens_objs[0]
    pp(l, 'lens saved object')
    info('Key fields:')
    info('  state.visualization  — chart type + layer configs')
    info('  state.datasourceStates.formBased.layers — field + operation definitions per layer')
    info('  state.filters        — global filters applied to this Lens')
    info('  state.query          — KQL query')
    info('references             — data view IDs used by each layer')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/visualize', 'Visualize Library — browse Lens visualizations')

In [ ]:
# Create a Lens bar chart via API (idempotent)
heading('4.4 Lens Visualization — create via API')

ecom_dv = next((o for o in find_saved_objects('index-pattern') if 'ecommerce' in o['attributes'].get('title', '')), None)
if not ecom_dv:
    raise RuntimeError('eCommerce data view not found — re-run 00_setup to reload sample data')
ecom_dv_id = ecom_dv['id']

# Reuse existing if already present
existing_lens = next((o for o in find_saved_objects('lens') if o['attributes'].get('title') == 'Training — Revenue by Day'), None)
if existing_lens:
    lens_id = existing_lens['id']
    info(f'Lens viz already exists: id={lens_id}')
else:
    lens_resp = kibana_post('/api/saved_objects/lens', {
        'attributes': {
            'title': 'Training — Revenue by Day',
            'visualizationType': 'lnsXY',
            'state': {
                'datasourceStates': {
                    'formBased': {
                        'layers': {
                            'layer1': {
                                'columnOrder': ['col_date', 'col_revenue'],
                                'columns': {
                                    'col_date': {
                                        'dataType': 'date',
                                        'isBucketed': True,
                                        'label': 'order_date',
                                        'operationType': 'date_histogram',
                                        'params': {'interval': '1d'},
                                        'scale': 'interval',
                                        'sourceField': 'order_date',
                                    },
                                    'col_revenue': {
                                        'dataType': 'number',
                                        'isBucketed': False,
                                        'label': 'Sum of taxful_total_price',
                                        'operationType': 'sum',
                                        'scale': 'ratio',
                                        'sourceField': 'taxful_total_price',
                                    },
                                },
                                'incompleteColumns': {},
                            }
                        }
                    }
                },
                'filters': [],
                'query': {'language': 'kuery', 'query': ''},
                'visualization': {
                    'layers': [{
                        'accessors': ['col_revenue'],
                        'layerId': 'layer1',
                        'layerType': 'data',
                        'seriesType': 'bar_stacked',
                        'xAccessor': 'col_date',
                        'yConfig': [],
                    }],
                    'legend': {'isVisible': True, 'position': 'right'},
                    'preferredSeriesType': 'bar_stacked',
                    'valueLabels': 'hide',
                },
            },
        },
        'references': [{
            'id': ecom_dv_id,
            'name': 'indexpattern-datasource-layer-layer1',
            'type': 'index-pattern',
        }],
    })
    lens_id = lens_resp['id']
    success(f'Lens viz created: id={lens_id}')

In [ ]:
heading('Lens saved object schema')
lens_obj = kibana_get(f'/api/saved_objects/lens/{lens_id}')
pp(lens_obj, 'lens saved object')
info('Key fields:')
info('  visualizationType                              — e.g. lnsXY, lnsMetric, lnsPie')
info('  state.datasourceStates.formBased.layers        — columns (fields + operations) per layer')
info('  state.visualization.layers                     — rendering config (series type, axes)')
info('  references                                     — data view IDs used by each layer')
kibana_link(f'/app/lens#/edit/{lens_id}', 'Open Training — Revenue by Day in Lens editor')

In [ ]:
# Ensure the Lens viz exists before snapshotting (makes cell re-runnable)
if not any(o['id'] == lens_id for o in find_saved_objects('lens')):
    warn(f'Lens viz {lens_id} missing — recreating before snapshot')
    ecom_dv_id = next(o['id'] for o in find_saved_objects('index-pattern') if 'ecommerce' in o['attributes'].get('title', ''))
    lens_resp = kibana_post('/api/saved_objects/lens', {
        'attributes': {
            'title': 'Training — Revenue by Day',
            'visualizationType': 'lnsXY',
            'state': {
                'datasourceStates': {'formBased': {'layers': {'layer1': {
                    'columnOrder': ['col_date', 'col_revenue'],
                    'columns': {
                        'col_date': {'dataType': 'date', 'isBucketed': True, 'label': 'order_date',
                            'operationType': 'date_histogram', 'params': {'interval': '1d'},
                            'scale': 'interval', 'sourceField': 'order_date'},
                        'col_revenue': {'dataType': 'number', 'isBucketed': False,
                            'label': 'Sum of taxful_total_price', 'operationType': 'sum',
                            'scale': 'ratio', 'sourceField': 'taxful_total_price'},
                    },
                    'incompleteColumns': {},
                }}}},
                'filters': [], 'query': {'language': 'kuery', 'query': ''},
                'visualization': {'layers': [{'accessors': ['col_revenue'], 'layerId': 'layer1',
                    'layerType': 'data', 'seriesType': 'bar_stacked', 'xAccessor': 'col_date', 'yConfig': []}],
                    'legend': {'isVisible': True, 'position': 'right'},
                    'preferredSeriesType': 'bar_stacked', 'valueLabels': 'hide'},
            },
        },
        'references': [{'id': ecom_dv_id, 'name': 'indexpattern-datasource-layer-layer1', 'type': 'index-pattern'}],
    })
    lens_id = lens_resp['id']
    success(f'Recreated Lens viz: id={lens_id}')

# Snapshot → accidentally delete → restore
snap_delete_restore_cycle('snap-lens-test', 'Lens Visualization')

kibana_delete(f'/api/saved_objects/lens/{lens_id}')
warn(f'Accidentally deleted Lens viz {lens_id}')
assert not any(o['id'] == lens_id for o in find_saved_objects('lens')), 'Should be gone'

restore_kibana_state(es, SNAP_REPO, 'snap-lens-test')
time.sleep(3)

restored = find_saved_objects('lens')
assert any(o['id'] == lens_id for o in restored), 'Lens viz should be restored'
success(f'Lens viz {lens_id} successfully restored!')
kibana_link(f'/app/lens#/edit/{lens_id}', 'Verify restored Lens viz in editor')